# A/B Testing & Causal Inference Report: Landing Page Redesign

**Project objective.** Evaluate whether a redesigned landing page improves conversion performance for an e-commerce business using the canonical Udacity `ab_data.csv` experiment log. The notebook is written as an executive-quality analysis that combines classical hypothesis testing, Bayesian decision analysis, segmentation, and business impact modeling.

**Notebook design principles.**
- Business-first framing: every statistical choice is tied back to a practical product decision.
- Transparent methodology: assumptions, trade-offs, and failure modes are stated explicitly.
- Reproducible analysis: all calculations are implemented in Python with clear comments and presentation-ready tables and charts.


## Section 1 — Business Context

A/B testing is the operational backbone of modern product development. In a digital product, nearly every customer-facing decision can be framed as a causal question: *if we change the experience, will user behavior improve?* Companies such as Netflix, Amazon, Google, Booking.com, Meta, and Airbnb run thousands of experiments every year because experimentation is one of the few scalable ways to learn what *causes* a business outcome to change rather than merely what correlates with it. Recommendation ranking, onboarding flows, pricing pages, search results, checkout design, messaging, notifications, and merchandising logic are all routinely tested because small causal gains compound into very large financial outcomes at internet scale.

In this project, we analyze a fictional but realistic e-commerce scenario. The company has redesigned its landing page and has randomly exposed some visitors to the **existing experience** (the control group) and others to the **new design** (the treatment group). Leadership wants a clear answer to a deceptively simple question: **does the new landing page increase conversions enough to justify a launch?**

A **conversion** is the business event the company ultimately cares about. In this context, we will define a conversion as a visitor taking the primary monetizable action on the site — for example, completing a purchase, submitting a qualified lead, or starting a paid subscription. Conversions are the bridge between product design and revenue. Even a small lift matters because scale magnifies percentage changes. If a business serves millions of visitors per year, a one-percent relative lift in conversion can translate into tens or hundreds of thousands of dollars in additional gross profit. That is why mature organizations treat experiment analysis as a capital allocation decision, not merely a reporting exercise.

The cost of making the wrong decision runs in both directions. If the company **launches a bad design**, it may reduce revenue, degrade customer trust, increase bounce rates, create downstream operational costs, and consume engineering capacity that could have been invested elsewhere. This is the cost of a false positive decision. On the other hand, if the company **rejects a genuinely better design**, it leaves growth on the table, delays learning, and may allow competitors to outperform it. This is the cost of a false negative decision. High-quality experimentation exists to manage this trade-off systematically: not to eliminate uncertainty, but to quantify it well enough for the business to make a rational decision.


## Section 2 — Experiment Design

At the heart of an A/B test is a comparison between two explanations of the world.

- The **null hypothesis** says the new page does **not** improve conversion relative to the old page in a meaningful way.
- The **alternative hypothesis** says the new page **does** improve conversion.

In plain English:
- **Null hypothesis (`H_0`)**: any observed lift is just random noise from sampling variability.
- **Alternative hypothesis (`H_1`)**: the treatment genuinely changes conversion and the lift is large enough to be actionable.

In statistical notation for a one-sided business question:
- `H_0: p_treatment <= p_control`
- `H_1: p_treatment > p_control`

Where `p_control` is the true conversion rate for the old page and `p_treatment` is the true conversion rate for the new page.

Two error types matter because both map to expensive product decisions:

- **Type I error (false positive):** we conclude the new page is better and launch it, but in reality it is not. In business terms, this means shipping a worse checkout, onboarding flow, or merchandising page and potentially harming revenue.
- **Type II error (false negative):** we conclude the new page is not better and keep the old one, but in reality the redesign would have improved conversion. In business terms, this means delaying growth and underinvesting in a beneficial product change.

We typically choose **`alpha = 0.05`** as the decision threshold for frequentist inference. This does **not** mean there is a five percent chance the null hypothesis is true. Instead, it means that *if the null hypothesis were actually true*, our testing procedure would falsely declare significance about five percent of the time in repeated experiments. It is a tolerance for false alarms, not a direct probability statement about the hypothesis.

**Statistical power** is the probability that the test will correctly detect a true effect of the size we care about. A power target of **0.80** means that if the new page really produces the minimum detectable lift we consider meaningful, we expect to detect it roughly 80% of the time. This is widely used because it creates a practical compromise: power that is high enough to avoid chronically underpowered experiments, but not so demanding that required sample sizes become operationally unrealistic.

Before looking at the data, we therefore ask a planning question: **how many users do we need per variant to have a fair chance of detecting a practically meaningful lift?**


In [5]:
# Import the sample size planning utility from statsmodels.  
from statsmodels.stats.power import NormalIndPower  # Estimate required sample sizes for two-sample proportion tests.

# Import the helper used to convert proportion differences into standardized effect sizes.  
from statsmodels.stats.proportion import proportion_effectsize  # Translate raw conversion deltas into Cohen's h.

# Set the assumed baseline conversion rate using a realistic e-commerce starting point.  
baseline_conversion_rate = 0.12  # Use 12% as a planning baseline because the Udacity dataset is near this level.

# Define the minimum detectable effect as a 2% relative lift over baseline.  
minimum_detectable_effect = baseline_conversion_rate * 0.02  # Convert a 2% relative improvement into absolute rate points.

# Calculate the treatment conversion rate implied by the minimum detectable effect.  
expected_treatment_rate = baseline_conversion_rate + minimum_detectable_effect  # Add the lift to the baseline rate.

# Set the false positive tolerance for the experiment.  
alpha = 0.05  # Use the conventional 5% significance level.

# Set the desired statistical power for the experiment.  
power = 0.80  # Aim for an 80% chance of detecting the target lift if it is real.

# Convert the business lift into the standardized effect size required by the power calculator.  
effect_size = proportion_effectsize(baseline_conversion_rate, expected_treatment_rate)  # Compute Cohen's h for two proportions.

# Create a power analysis object for a two-sample z-test on proportions.  
power_analysis = NormalIndPower()  # Instantiate the solver for independent group comparisons.

# Solve for the number of observations needed in each group.
# Use abs() and float() to extract a clean scalar from the result, which can be a numpy array in some statsmodels versions.
raw_sample_size = power_analysis.solve_power(
    effect_size=abs(float(effect_size)),
    power=power,
    alpha=alpha,
    ratio=1.0,
    alternative='larger'
)  # Assume equal allocation and a one-sided business hypothesis.

# Force the result to a plain Python float so f-string formatting works reliably.  
required_sample_size_per_group = float(raw_sample_size)  # Ensure a scalar, not a numpy array.

# Round the sample size up because partial users are not meaningful in production.  
required_sample_size_per_group_rounded = int(required_sample_size_per_group) + (1 if required_sample_size_per_group % 1 > 0 else 0)  # Ensure the recommendation is conservative.

# Calculate the total required sample size across both variants.  
total_required_sample_size = required_sample_size_per_group_rounded * 2  # Multiply by two because we have control and treatment.

# Print a business-friendly explanation of the planning result.  
print(f"With a baseline conversion rate of {baseline_conversion_rate:.2%} and a minimum detectable lift of {minimum_detectable_effect:.2%} ({0.02:.0%} relative), the experiment needs about {required_sample_size_per_group_rounded:,} users per group, or {total_required_sample_size:,} users in total, to achieve {power:.0%} power at alpha = {alpha:.2f}.")

# Print an interpretation that ties sample size back to decision quality.  
print(f"In practical terms, if the true lift is at least {minimum_detectable_effect:.2%} absolute, an experiment of this size gives the company a strong chance of detecting it without accepting an excessive false-positive risk.")

With a baseline conversion rate of 12.00% and a minimum detectable lift of 0.24% (2% relative), the experiment needs about 228,644 users per group, or 457,288 users in total, to achieve 80% power at alpha = 0.05.
In practical terms, if the true lift is at least 0.24% absolute, an experiment of this size gives the company a strong chance of detecting it without accepting an excessive false-positive risk.


## Section 3 — Data Loading & Exploration

Each row in `ab_data.csv` represents a user-level exposure to the experiment.

- **`user_id`**: the unique identifier for the visitor. In business terms, this is the unit of experimentation. We want each user to contribute independent evidence, not repeated exposures that blur attribution.
- **`timestamp`**: when the exposure occurred. This is operationally important because experiment performance can drift over time due to seasonality, novelty effects, marketing campaigns, or engineering incidents.
- **`group`**: the assigned experimental condition. `control` users are intended to see the existing page, while `treatment` users are intended to see the redesigned page.
- **`landing_page`**: the actual page the user saw. This is the realized experience, which should align with the assigned group if randomization and instrumentation worked correctly.
- **`converted`**: whether the user completed the target action. This is the binary business outcome we are trying to improve.

Before running formal inference, we validate the integrity of the experiment log. Data quality is not a cosmetic concern in experimentation — it is part of causal validity. If a user appears in conflicting variants or assignment does not match exposure, we can no longer cleanly interpret the test as randomized evidence. That is why we explicitly check for duplicates, page-assignment mismatches, and group balance.


In [7]:
# Import the core analysis libraries used throughout the notebook.  
import numpy as np  # Support vectorized numerical calculations and Bayesian simulation.
import pandas as pd  # Load, clean, aggregate, and summarize the experiment dataset.
import plotly.express as px  # Create concise publication-ready exploratory charts.
import plotly.graph_objects as go  # Build custom statistical visualizations such as confidence intervals.

# Define the path to the expected Udacity dataset.  
data_path = 'data/ab_data.csv'  # Keep the filename explicit so the notebook is easy to reuse.

# Load the experiment log into a pandas DataFrame.  
df_raw = pd.read_csv(data_path)  # Read the CSV file from the working directory.

# Display the dataset shape so we understand the raw experiment volume.  
print(f"Raw dataset shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")  # Summarize the raw size of the table.

# Show data types to confirm whether each field is being interpreted correctly.  
print('\nColumn types:')  # Add a label before printing data types.
print(df_raw.dtypes)  # Inspect the inferred schema for the raw dataset.

# Preview the first few rows to sanity check column values.  
print('\nFirst five rows:')  # Add a label before printing the preview.
print(df_raw.head())  # Inspect the first records in the dataset.

# Display descriptive statistics for both numeric and categorical columns.  
print('\nDescriptive summary:')  # Add a label before printing summary statistics.
print(df_raw.describe(include='all'))  # Review top-level distributions and value counts.

# Count exact duplicate rows across all columns.  
exact_duplicate_rows = df_raw.duplicated().sum()  # Measure redundant records that are identical in every field.

# Count duplicate user identifiers because repeat users can violate unit-of-analysis assumptions.  
duplicate_user_ids = df_raw['user_id'].duplicated().sum()  # Identify users who appear more than once in the experiment log.

# Print the duplicate diagnostics in business terms.  
print(f"\nExact duplicate rows: {exact_duplicate_rows:,}")  # Report perfect row duplication.
print(f"Duplicate user IDs: {duplicate_user_ids:,}")  # Report repeated users that may require adjudication.

# Remove exact duplicate rows first because they contribute no additional information.  
df = df_raw.drop_duplicates().copy()  # Create a cleaned working copy after dropping exact duplicates.

# Create a boolean mask for assignment mismatches between group and landing page.  
mismatch_mask = ((df['group'] == 'treatment') & (df['landing_page'] != 'new_page')) | ((df['group'] == 'control') & (df['landing_page'] != 'old_page'))  # Identify rows where random assignment and actual exposure conflict.

# Count the number of mismatched records before dropping them.  
mismatch_count = int(mismatch_mask.sum())  # Convert the count to an integer for clean reporting.

# Print why mismatches are a causal inference problem.  
print(f"Mismatched assignment/exposure rows: {mismatch_count:,}")  # Quantify instrumentation or logging inconsistencies.
print("These rows are removed because they break the clean interpretation that assignment maps to experience, which is essential for unbiased experimental estimation.")  # Explain the causal integrity issue.

# Remove mismatched rows to preserve a valid control-versus-treatment comparison.  
df = df.loc[~mismatch_mask].copy()  # Keep only rows where assignment and exposure agree.

# Drop duplicate users by keeping the first observed exposure for each user.  
df = df.drop_duplicates(subset='user_id', keep='first').copy()  # Ensure each user contributes at most one experimental observation.

# Convert timestamps to pandas datetime values for time-based analysis.  
df['timestamp'] = pd.to_datetime(df['timestamp'])  # Parse the timestamp strings into datetime objects.

# Print the final cleaned dataset shape.  
print(f"\nCleaned dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")  # Summarize the analysis-ready dataset.

# Calculate group sizes after cleaning.  
group_sizes = df['group'].value_counts().rename_axis('group').reset_index(name='users')  # Count the number of users in each experiment arm.

# Calculate the absolute and relative imbalance between groups.  
max_group_size = group_sizes['users'].max()  # Identify the larger group size.
min_group_size = group_sizes['users'].min()  # Identify the smaller group size.
absolute_imbalance = int(max_group_size - min_group_size)  # Compute the difference in user counts.
relative_imbalance = absolute_imbalance / df.shape[0]  # Express the imbalance as a share of the cleaned sample.

# Print a short balance diagnostic.  
print(f"Absolute group size difference: {absolute_imbalance:,} users")  # Report the raw difference in traffic allocation.
print(f"Relative group size difference: {relative_imbalance:.4%}")  # Report the size gap as a fraction of the full sample.
print("Balanced groups matter because unequal allocation can reduce power and, if caused by logging issues, may indicate randomization problems rather than mere chance.")  # Explain why balance is important operationally.

# Plot the cleaned group sizes with Plotly for a fast traffic-allocation visual check.  
fig_group_sizes = px.bar(group_sizes, x='group', y='users', color='group', text='users', title='Control vs Treatment Group Sizes After Data Cleaning', labels={'group': 'Experiment Group', 'users': 'Number of Users'})  # Create a bar chart comparing traffic by experimental arm.

# Improve the chart formatting for presentation use.  
fig_group_sizes.update_traces(texttemplate='%{text:,}', textposition='outside')  # Display user counts above each bar.
fig_group_sizes.update_layout(showlegend=False)  # Remove the redundant legend because color already matches axis labels.

# Render the chart inline in the notebook.  
fig_group_sizes.show()  # Display the traffic allocation chart.


Raw dataset shape: 294,478 rows x 5 columns

Column types:
user_id          int64
timestamp       object
group           object
landing_page    object
converted        int64
dtype: object

First five rows:
   user_id                   timestamp      group landing_page  converted
0   851104  2017-01-21 22:11:48.556739    control     old_page          0
1   804228  2017-01-12 08:01:45.159739    control     old_page          0
2   661590  2017-01-11 16:55:06.154213  treatment     new_page          0
3   853541  2017-01-08 18:28:03.143765  treatment     new_page          0
4   864975  2017-01-21 01:52:26.210827    control     old_page          1

Descriptive summary:
              user_id                   timestamp      group landing_page  \
count   294478.000000                      294478     294478       294478   
unique            NaN                      294478          2            2   
top               NaN  2017-01-21 22:11:48.556739  treatment     old_page   
freq              Na

**Chart caption.** The bar chart should show whether traffic allocation remained close to 50/50 after data cleaning. Mild differences are expected from random assignment, but large imbalances can signal implementation or logging issues that weaken confidence in the experiment.


## Section 4 — Exploratory Data Analysis

Before running formal statistical tests, we want to understand the experiment at a descriptive level. The goal is not to "peek" our way into a decision, but to identify patterns that matter for interpretation:

1. **What are the raw conversion rates by group?** This tells us the observed direction and magnitude of the lift.
2. **How stable are conversion rates over time?** A treatment that looks strong early but decays later may reflect novelty rather than durable product value.
3. **Did the experiment run long enough?** Even a well-randomized experiment can produce misleading results if it ends before capturing normal day-of-week variation or enough sample size to reach the planned power target.

EDA is the bridge between instrumentation checks and formal inference. It helps us distinguish between a genuine treatment effect, normal sampling noise, and transient operational effects.


In [8]:
# Calculate conversion summaries for each experimental group.  
conversion_summary = df.groupby('group').agg(users=('user_id', 'nunique'), conversions=('converted', 'sum'), conversion_rate=('converted', 'mean')).reset_index()  # Aggregate sample size, conversions, and rates by group.

# Print the raw conversion summary table.  
print(conversion_summary)  # Display the descriptive conversion metrics.

# Create a calendar date column for daily trend analysis.  
df['event_date'] = df['timestamp'].dt.date  # Collapse timestamps to the date level for time-series visualization.

# Aggregate daily conversion performance for each group.  
daily_conversion = df.groupby(['event_date', 'group']).agg(users=('user_id', 'nunique'), conversions=('converted', 'sum')).reset_index()  # Summarize daily traffic and conversion counts.

# Calculate the daily conversion rate for plotting.  
daily_conversion['conversion_rate'] = daily_conversion['conversions'] / daily_conversion['users']  # Convert counts into daily percentages.

# Plot the daily conversion rate trajectories for control and treatment.  
fig_daily = px.line(daily_conversion, x='event_date', y='conversion_rate', color='group', markers=True, title='Daily Conversion Rates Over Time by Experiment Group', labels={'event_date': 'Date', 'conversion_rate': 'Daily Conversion Rate', 'group': 'Experiment Group'})  # Create a time-series view of daily experiment performance.

# Format the y-axis as percentages for easier business interpretation.  
fig_daily.update_yaxes(tickformat='.2%')  # Express conversion rates as percentages.

# Render the daily trend chart.  
fig_daily.show()  # Display the temporal conversion-rate comparison.

# Identify the experiment start and end dates.  
experiment_start = df['timestamp'].min()  # Capture the earliest exposure timestamp.
experiment_end = df['timestamp'].max()  # Capture the latest exposure timestamp.

# Calculate the experiment duration in days.  
experiment_duration_days = (experiment_end - experiment_start).days + 1  # Add one so inclusive dates map to full-day duration.

# Print the timing diagnostics for duration review.  
print(f"Experiment start: {experiment_start}")  # Report the first timestamp in the cleaned data.
print(f"Experiment end: {experiment_end}")  # Report the last timestamp in the cleaned data.
print(f"Experiment duration: {experiment_duration_days} days")  # Report how long the experiment ran.

# Reuse the planning assumption from Section 2 to check whether the experiment reached the target sample size.  
achieved_sample_per_group = conversion_summary['users'].min()  # Use the smaller group as the binding sample-size constraint.

# Print whether the experiment meets the planned sample threshold.  
print(f"Minimum achieved users in a group: {achieved_sample_per_group:,}")  # Report the smaller group size after cleaning.
print(f"Required users per group from planning: {required_sample_size_per_group_rounded:,}")  # Remind the reader of the planned sample requirement.
print(f"Did the experiment meet the planned minimum? {'Yes' if achieved_sample_per_group >= required_sample_size_per_group_rounded else 'No'}")  # State whether the observed sample appears sufficient.


       group   users  conversions  conversion_rate
0    control  145274        17489         0.120386
1  treatment  145310        17264         0.118808


Experiment start: 2017-01-02 13:42:05.378582
Experiment end: 2017-01-24 13:41:54.460509
Experiment duration: 22 days
Minimum achieved users in a group: 145,274
Required users per group from planning: 228,644
Did the experiment meet the planned minimum? No


### Novelty Effect: Why Early Results Can Mislead

A common pattern in product experiments is the **novelty effect**. Users often react positively to something simply because it is new, visually different, or temporarily more attention-grabbing — not because it is fundamentally better over the long run. This can produce a short-lived lift in metrics such as click-through rate or conversion during the first few days of an experiment, followed by reversion toward baseline once users adapt.

In practice, this matters because leadership decisions are usually permanent while novelty effects are temporary. If the treatment curve starts above the control curve and then converges or declines, that is a warning sign that the design may be capturing curiosity rather than creating durable value. A mature experiment review therefore looks at temporal stability, not just the final aggregate p-value.


## Section 5 — Frequentist A/B Test

A **two-proportion z-test** is the standard frequentist tool for comparing conversion rates in two independent groups. In plain English, it asks: *if the control and treatment truly had the same underlying conversion rate, how unusual would the observed difference be purely due to random sampling noise?*

The output most people focus on is the **p-value**, but it is often misunderstood. A p-value is **not** the probability that the null hypothesis is true. It is also **not** the probability that the result happened by chance in some vague everyday sense. More precisely, the p-value is the probability of observing a result at least as extreme as the one in our sample, **assuming the null hypothesis were true**.

That distinction matters. A small p-value tells us the data would be surprising under the null. It does not directly tell us how large the effect is, whether the effect is practically valuable, or how likely the treatment is to win in the future.

That is why we also calculate a **95% confidence interval** for the conversion-rate difference. The confidence interval gives a plausible range of effect sizes consistent with the observed data under the frequentist framework. It answers a question the p-value does not: **what sizes of lift or decline remain compatible with the evidence?** A decision-maker should care deeply about this because a result can be statistically non-significant yet still leave open the possibility of materially large gains or losses if the sample is too small.


In [9]:
# Import the frequentist testing utilities from statsmodels.  
from statsmodels.stats.proportion import confint_proportions_2indep  # Compute a confidence interval for the difference in two proportions.
from statsmodels.stats.proportion import proportions_ztest  # Run the classical z-test for two independent conversion rates.

# Extract control-group conversions and sample size.  
control_conversions = int(df.loc[df['group'] == 'control', 'converted'].sum())  # Count converted users in the control group.
control_users = int(df.loc[df['group'] == 'control', 'user_id'].nunique())  # Count unique users in the control group.

# Extract treatment-group conversions and sample size.  
treatment_conversions = int(df.loc[df['group'] == 'treatment', 'converted'].sum())  # Count converted users in the treatment group.
treatment_users = int(df.loc[df['group'] == 'treatment', 'user_id'].nunique())  # Count unique users in the treatment group.

# Run the two-proportion z-test using a one-sided business hypothesis.  
z_statistic, p_value = proportions_ztest([treatment_conversions, control_conversions], [treatment_users, control_users], alternative='larger')  # Test whether treatment converts better than control.

# Compute the observed conversion rates for interpretation.  
control_rate = control_conversions / control_users  # Calculate the control conversion rate.
treatment_rate = treatment_conversions / treatment_users  # Calculate the treatment conversion rate.
observed_lift = treatment_rate - control_rate  # Measure the absolute difference in conversion rates.
relative_lift = observed_lift / control_rate  # Express the lift relative to the control baseline.

# Calculate the 95% confidence interval for the difference in proportions.  
ci_low, ci_high = confint_proportions_2indep(count1=treatment_conversions, nobs1=treatment_users, count2=control_conversions, nobs2=control_users, compare='diff', alpha=0.05, method='wald')  # Estimate a 95% confidence interval for treatment minus control.

# Print the key frequentist outputs.  
print(f"Control conversion rate: {control_rate:.4%}")  # Report the control baseline.
print(f"Treatment conversion rate: {treatment_rate:.4%}")  # Report the treatment performance.
print(f"Observed absolute lift: {observed_lift:.4%}")  # Report the treatment minus control difference.
print(f"Observed relative lift: {relative_lift:.2%}")  # Report the relative change versus the control baseline.
print(f"Z-statistic: {z_statistic:.4f}")  # Report the test statistic.
print(f"One-sided p-value: {p_value:.4f}")  # Report the frequentist significance measure.
print(f"95% confidence interval for the lift: [{ci_low:.4%}, {ci_high:.4%}]")  # Report the plausible effect-size range.

# Build a compact DataFrame for confidence-interval plotting.  
ci_df = pd.DataFrame({'metric': ['Treatment minus Control'], 'estimate': [observed_lift], 'ci_low': [ci_low], 'ci_high': [ci_high]})  # Package the estimate and interval into a tidy structure.

# Create a horizontal confidence-interval chart.  
fig_ci = go.Figure()  # Initialize a custom Plotly figure for the interval visualization.

# Add the point estimate with error bars.  
fig_ci.add_trace(go.Scatter(x=ci_df['estimate'], y=ci_df['metric'], mode='markers', marker=dict(size=12, color='#1f77b4'), error_x=dict(type='data', symmetric=False, array=ci_df['ci_high'] - ci_df['estimate'], arrayminus=ci_df['estimate'] - ci_df['ci_low']), name='Estimated Lift'))  # Plot the lift and its confidence interval.

# Add a vertical reference line at zero effect.  
fig_ci.add_vline(x=0, line_dash='dash', line_color='firebrick')  # Show the no-difference benchmark.

# Format the chart for executive readability.  
fig_ci.update_layout(title='95% Confidence Interval for the Conversion Lift', xaxis_title='Lift in Conversion Rate (Treatment - Control)', yaxis_title='Comparison', showlegend=False)  # Add presentation-ready titles and axis labels.
fig_ci.update_xaxes(tickformat='.2%')  # Display the effect size as percentages.

# Render the confidence-interval chart.  
fig_ci.show()  # Display the interval estimate visually.


Control conversion rate: 12.0386%
Treatment conversion rate: 11.8808%
Observed absolute lift: -0.1578%
Observed relative lift: -1.31%
Z-statistic: -1.3109
One-sided p-value: 0.9051
95% confidence interval for the lift: [-0.3938%, 0.0781%]


**Chart caption.** The dot represents the observed conversion lift, while the horizontal bar represents the 95% confidence interval. If the interval crosses zero, the data remain compatible with both a small gain and a small loss, which weakens the case for a confident launch.


### Frequentist Interpretation in Business Language

For the canonical Udacity landing-page dataset, the treatment effect is typically **small and statistically unconvincing** once the data are cleaned correctly. In business terms, that means the redesign does **not** clear the evidentiary bar required for a confident launch. The observed lift is too small relative to the noise in the experiment, and the confidence interval usually includes zero or values close enough to zero that the practical upside is questionable.

A prudent recommendation from the frequentist analysis is therefore: **do not launch solely on the basis of this test**. Either keep the current page, or continue experimentation with a stronger design change, a longer run time, or richer instrumentation that can explain *why* the redesign did not produce a measurable gain.


## Section 6 — Bayesian A/B Test

Frequentist and Bayesian approaches answer related but importantly different questions.

A useful analogy is weather forecasting. A frequentist test is like asking, *"If there were truly no difference between the pages, how surprising would today's observed data be?"* A Bayesian analysis is like asking, *"Given what we believed before and what we have now observed, what should we believe about the treatment's conversion rate today?"* The Bayesian framing is often more natural for decision-making because businesses usually care about probabilities of outcomes and the costs of decisions, not just rejection thresholds.

Three Bayesian ingredients matter conceptually:

- **Prior:** what we believe about the conversion rate before seeing this experiment. A weak prior can simply encode that all rates between 0 and 1 are initially plausible.
- **Likelihood:** how probable the observed data are for different possible conversion rates.
- **Posterior:** the updated belief after combining the prior with the observed evidence.

For binary outcomes such as conversion, the Beta distribution is a natural prior and posterior family. With a simple Beta prior, we can simulate the posterior distribution of each group's conversion rate and ask directly useful business questions:

- What is the probability the treatment beats the control?
- If we choose the wrong page, how much conversion do we expect to lose?

Companies such as Booking.com and Airbnb often favor Bayesian decision support because it aligns naturally with iterative product decisions, sequential learning, and expected-value thinking. Bayesian outputs also tend to be easier for non-technical stakeholders to interpret because probabilities like *"the treatment has a 23% chance of being better"* are closer to the way businesses already reason.


In [10]:
# Set the random seed so the posterior simulation is reproducible.  
np.random.seed(42)  # Fix the seed for stable Monte Carlo results.

# Choose a weakly informative Beta(1, 1) prior for each variant.  
prior_alpha = 1  # Start with one pseudo-conversion for the prior.
prior_beta = 1  # Start with one pseudo-non-conversion for the prior.

# Define the number of Monte Carlo samples for posterior simulation.  
posterior_draws = 200_000  # Use a large sample to stabilize posterior probability estimates.

# Simulate the posterior conversion-rate distribution for the control group.  
control_posterior = np.random.beta(prior_alpha + control_conversions, prior_beta + control_users - control_conversions, posterior_draws)  # Draw samples from the control posterior.

# Simulate the posterior conversion-rate distribution for the treatment group.  
treatment_posterior = np.random.beta(prior_alpha + treatment_conversions, prior_beta + treatment_users - treatment_conversions, posterior_draws)  # Draw samples from the treatment posterior.

# Compute the posterior distribution of the treatment lift.  
posterior_lift = treatment_posterior - control_posterior  # Subtract control from treatment draw by draw.

# Estimate the probability that treatment is better than control.  
prob_treatment_better = float((posterior_lift > 0).mean())  # Measure the share of posterior draws where treatment wins.

# Estimate expected loss if we choose treatment when control is actually better.  
expected_loss_choose_treatment = float(np.maximum(control_posterior - treatment_posterior, 0).mean())  # Average the regret when treatment loses.

# Estimate expected loss if we choose control when treatment is actually better.  
expected_loss_choose_control = float(np.maximum(treatment_posterior - control_posterior, 0).mean())  # Average the regret when control loses.

# Build a common x-axis for plotting the posterior densities.  
x_grid = np.linspace(min(control_posterior.min(), treatment_posterior.min()), max(control_posterior.max(), treatment_posterior.max()), 500)  # Span the support of both posterior distributions.

# Estimate smooth posterior densities using histogram-based approximations.  
control_density, control_bins = np.histogram(control_posterior, bins=200, density=True)  # Approximate the control posterior density.
treatment_density, treatment_bins = np.histogram(treatment_posterior, bins=200, density=True)  # Approximate the treatment posterior density.

# Convert histogram bin edges to midpoints for plotting.  
control_midpoints = (control_bins[:-1] + control_bins[1:]) / 2  # Compute the control density x-values.
treatment_midpoints = (treatment_bins[:-1] + treatment_bins[1:]) / 2  # Compute the treatment density x-values.

# Initialize the posterior comparison chart.  
fig_bayes = go.Figure()  # Create a figure for the posterior density comparison.

# Add the control posterior with a shaded area.  
fig_bayes.add_trace(go.Scatter(x=control_midpoints, y=control_density, mode='lines', name='Control Posterior', line=dict(color='#1f77b4'), fill='tozeroy', opacity=0.45))  # Plot the control posterior density.

# Add the treatment posterior with a shaded area.  
fig_bayes.add_trace(go.Scatter(x=treatment_midpoints, y=treatment_density, mode='lines', name='Treatment Posterior', line=dict(color='#2ca02c'), fill='tozeroy', opacity=0.45))  # Plot the treatment posterior density.

# Format the Bayesian chart.  
fig_bayes.update_layout(title='Bayesian Posterior Distributions for Control and Treatment Conversion Rates', xaxis_title='Conversion Rate', yaxis_title='Posterior Density')  # Add publication-ready titles.
fig_bayes.update_xaxes(tickformat='.2%')  # Format the conversion-rate axis as percentages.

# Render the posterior comparison chart.  
fig_bayes.show()  # Display the Bayesian posterior densities.

# Print the Bayesian decision metrics.  
print(f"Posterior probability that treatment beats control: {prob_treatment_better:.2%}")  # Report the Bayesian win probability.
print(f"Expected loss if we launch treatment and it is worse: {expected_loss_choose_treatment:.4%} conversion points")  # Report regret for choosing treatment.
print(f"Expected loss if we keep control and treatment is actually better: {expected_loss_choose_control:.4%} conversion points")  # Report regret for choosing control.


Posterior probability that treatment beats control: 9.59%
Expected loss if we launch treatment and it is worse: 0.1637% conversion points
Expected loss if we keep control and treatment is actually better: 0.0054% conversion points


**Chart caption.** The more the green treatment posterior sits to the right of the blue control posterior, the stronger the Bayesian evidence that the redesign improves conversion. Substantial overlap implies material uncertainty even if one variant has a slightly higher sample mean.


### Comparing Bayesian and Frequentist Conclusions

For this dataset, the Bayesian conclusion will usually align with the frequentist one: the treatment does not demonstrate a compelling enough advantage to justify a confident launch. The Bayesian framing, however, adds two useful decision quantities that the frequentist test does not provide directly: the **probability of winning** and the **expected loss from a wrong decision**.

When Bayesian and frequentist conclusions disagree, the reason is usually not that one method is "right" and the other is "wrong." Instead, they are answering different questions. A frequentist result may fail to reject the null because the sample is noisy, while a Bayesian analysis may still show a moderate probability that treatment is better. Conversely, a result can cross a frequentist significance threshold while Bayesian expected loss remains too high for a launch decision. In practice, disagreement is often a signal to slow down and ask whether the effect is stable, large enough to matter, and robust to prior assumptions.


## Section 7 — Segmentation Analysis

Aggregate experiment results can hide important heterogeneity. A redesign that has no clear effect overall may still help some user segments and hurt others. If we look only at the top-line average, we can miss actionable patterns that matter for merchandising, personalization, device strategy, or rollout planning.

This connects to **Simpson's Paradox**, where an overall trend reverses or disappears after data are broken into meaningful subgroups. A classic example appears in university admissions analyses: a school can look biased against a group in the aggregate even when each department shows the opposite pattern, because applicants are distributed unevenly across departments with different selectivity levels. In experimentation, the same logic applies when traffic mix, geography, device type, or timing differs across subpopulations.

Because the Udacity dataset includes timestamps, we segment by **day of week** to check whether the redesign performs differently across temporal cohorts. This is a simple but realistic diagnostic because user intent and purchasing behavior often differ between weekdays and weekends.


In [11]:
# Initialize an empty placeholder for segmentation results.  
segment_results = pd.DataFrame()  # Create the object up front so later business-impact logic can reference it safely.

# Check whether the cleaned dataset includes a timestamp column for temporal segmentation.  
if 'timestamp' in df.columns:  # Proceed only if time data are available.
    # Create a day-of-week segment label from the timestamp.  
    df['day_of_week'] = df['timestamp'].dt.day_name()  # Convert timestamps into readable weekday names.

    # Aggregate conversion performance by day of week and experiment group.  
    segment_conversion = df.groupby(['day_of_week', 'group']).agg(users=('user_id', 'nunique'), conversions=('converted', 'sum')).reset_index()  # Summarize traffic and outcomes within each time segment.

    # Calculate conversion rates within each segment.  
    segment_conversion['conversion_rate'] = segment_conversion['conversions'] / segment_conversion['users']  # Express segment performance as percentages.

    # Define a business-friendly weekday order for plotting.  
    weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']  # Preserve the natural calendar order.

    # Cast day-of-week labels to an ordered categorical type.  
    segment_conversion['day_of_week'] = pd.Categorical(segment_conversion['day_of_week'], categories=weekday_order, ordered=True)  # Ensure the line chart follows the week chronologically.

    # Sort the segment table after applying the categorical order.  
    segment_conversion = segment_conversion.sort_values(['day_of_week', 'group'])  # Prepare a tidy ordered table for plotting and review.

    # Plot segmented conversion rates by day of week.  
    fig_segments = px.line(segment_conversion, x='day_of_week', y='conversion_rate', color='group', markers=True, title='Conversion Rate by Day of Week and Experiment Group', labels={'day_of_week': 'Day of Week', 'conversion_rate': 'Conversion Rate', 'group': 'Experiment Group'})  # Create the day-of-week segment chart.

    # Format the y-axis as percentages for readability.  
    fig_segments.update_yaxes(tickformat='.2%')  # Display segment conversion rates as percentages.

    # Render the segmentation chart.  
    fig_segments.show()  # Display the day-of-week segment performance chart.

    # Initialize a list to collect one z-test result per weekday.  
    segment_test_rows = []  # Store each segment's sample sizes, rates, and significance metrics.

    # Loop through each weekday segment in calendar order.  
    for day_name in weekday_order:  # Evaluate each temporal segment separately.
        # Subset the control rows for the current weekday.  
        control_segment = df[(df['group'] == 'control') & (df['day_of_week'] == day_name)]  # Filter control users within the current segment.

        # Subset the treatment rows for the current weekday.  
        treatment_segment = df[(df['group'] == 'treatment') & (df['day_of_week'] == day_name)]  # Filter treatment users within the current segment.

        # Skip segments that do not contain both groups.  
        if control_segment.empty or treatment_segment.empty:  # Avoid invalid tests in missing-data segments.
            continue  # Move to the next segment when one arm has no data.

        # Count conversions and users for the control segment.  
        control_segment_conversions = int(control_segment['converted'].sum())  # Count conversions in the control subset.
        control_segment_users = int(control_segment['user_id'].nunique())  # Count users in the control subset.

        # Count conversions and users for the treatment segment.  
        treatment_segment_conversions = int(treatment_segment['converted'].sum())  # Count conversions in the treatment subset.
        treatment_segment_users = int(treatment_segment['user_id'].nunique())  # Count users in the treatment subset.

        # Run a one-sided z-test for the current segment.  
        segment_z, segment_p = proportions_ztest([treatment_segment_conversions, control_segment_conversions], [treatment_segment_users, control_segment_users], alternative='larger')  # Test whether treatment outperforms control within the segment.

        # Compute segment-level conversion rates.  
        control_segment_rate = control_segment_conversions / control_segment_users  # Calculate the control conversion rate for the segment.
        treatment_segment_rate = treatment_segment_conversions / treatment_segment_users  # Calculate the treatment conversion rate for the segment.

        # Append the segment result as a dictionary.  
        segment_test_rows.append({'day_of_week': day_name, 'control_users': control_segment_users, 'treatment_users': treatment_segment_users, 'control_rate': control_segment_rate, 'treatment_rate': treatment_segment_rate, 'lift': treatment_segment_rate - control_segment_rate, 'z_statistic': segment_z, 'p_value': segment_p, 'significant_at_05': segment_p < 0.05})  # Capture the core metrics for the segment-level decision table.

    # Convert the collected dictionaries into a formatted DataFrame.  
    segment_results = pd.DataFrame(segment_test_rows)  # Build the final segmentation results table.

    # Apply percentage formatting-friendly rounding for human review.  
    display_columns = ['day_of_week', 'control_users', 'treatment_users', 'control_rate', 'treatment_rate', 'lift', 'z_statistic', 'p_value', 'significant_at_05']  # Define the visible column order.

    # Print the segmentation test table.  
    print(segment_results[display_columns].to_string(index=False))  # Display the segment-level comparison table.

    # Check whether any segment is significant when the aggregate result is not.  
    contradictory_segments = segment_results[segment_results['significant_at_05']]  # Identify segments with locally significant differences.

    # Print the segmentation take-away.  
    if contradictory_segments.empty:  # Handle the common case where no segment stands out.
        print("\nNo day-of-week segment shows a statistically significant treatment win at the 5% level that overturns the aggregate story.")  # State that segmentation does not materially change the headline decision.
    else:  # Handle the case where one or more segments appear locally significant.
        print(f"\n{contradictory_segments.shape[0]} segment(s) show a statistically significant treatment lift that deserves follow-up before a blanket launch decision.")  # Highlight segments that may warrant targeted rollout or additional analysis.
else:  # Provide a graceful fallback when timestamps are unavailable.
    print("No timestamp column is available, so day-of-week segmentation cannot be performed on this dataset.")  # Explain why the segmentation step is being skipped.


day_of_week  control_users  treatment_users  control_rate  treatment_rate      lift  z_statistic  p_value  significant_at_05
     Monday          22794            22646      0.122795        0.119491 -0.003304    -1.079275 0.859768              False
    Tuesday          23615            23533      0.116748        0.122339  0.005591     1.871037 0.030670               True
  Wednesday          19748            19817      0.121835        0.118837 -0.002998    -0.916367 0.820263              False
   Thursday          19527            19694      0.121729        0.118209 -0.003520    -1.072836 0.858328              False
     Friday          19692            19934      0.115834        0.117538  0.001704     0.528269 0.298656              False
   Saturday          19901            19768      0.124567        0.117058 -0.007509    -2.294268 0.989112              False
     Sunday          19997            19918      0.119518        0.117431 -0.002086    -0.644931 0.740514              False


**Chart caption.** Segment-level charts help reveal whether the treatment is consistently better, consistently worse, or unstable across time slices. A launch decision is stronger when the treatment performs coherently across segments rather than relying on isolated wins.


### Segment Interpretation for the Business Decision

If segmentation shows no material contradiction to the aggregate result, leadership can be more confident that the redesign simply does not create a broad enough conversion benefit to justify launch. If, however, a specific segment looks strong while the aggregate does not, the right response is usually **not** to declare victory immediately. Instead, the business should ask whether that segment result is theoretically plausible, operationally important, and robust to multiple-testing concerns.

A good business follow-up might be to run a targeted confirmatory experiment, personalize the experience for the promising segment, or instrument richer user attributes that explain why the redesign resonates differently across traffic cohorts.


## Section 8 — Business Impact Quantification

Statistical significance is necessary for disciplined decision-making, but it is not sufficient. A result can be statistically significant and still be too small to matter financially, especially at modest traffic levels. Conversely, a result can be statistically inconclusive yet economically interesting if the upside is large and the cost of delay is meaningful.

That is why the last step in an experiment review is to convert the estimated lift into money. Here we translate the observed conversion difference into projected incremental orders and revenue, then compare that upside to the hypothetical implementation cost of the redesign.


In [12]:
# Define the business assumptions provided in the project brief.  
monthly_visitors = 50_000  # Assume the site receives fifty thousand visitors each month.
average_order_value = 35.00  # Assume each converting order is worth thirty-five dollars on average.
implementation_cost = 20_000.00  # Assume the redesign cost twenty thousand dollars to build.

# Convert the observed conversion lift into incremental monthly orders.  
projected_incremental_orders_monthly = monthly_visitors * observed_lift  # Multiply monthly traffic by the measured absolute lift.

# Translate incremental orders into monthly revenue impact.  
projected_monthly_revenue_impact = projected_incremental_orders_monthly * average_order_value  # Convert extra conversions into dollars.

# Translate monthly impact into annual revenue impact.  
projected_annual_revenue_impact = projected_monthly_revenue_impact * 12  # Scale the revenue change to a twelve-month horizon.

# Calculate the break-even period in months when the revenue impact is positive.  
break_even_months = implementation_cost / projected_monthly_revenue_impact if projected_monthly_revenue_impact > 0 else np.inf  # Estimate how long it takes for the redesign to pay back its cost.

# Build a clean business-impact summary table.  
business_impact_table = pd.DataFrame({'metric': ['Observed absolute lift', 'Observed relative lift', 'Projected incremental orders / month', 'Projected revenue impact / month', 'Projected revenue impact / year', 'Implementation cost', 'Break-even period (months)'], 'value': [observed_lift, relative_lift, projected_incremental_orders_monthly, projected_monthly_revenue_impact, projected_annual_revenue_impact, implementation_cost, break_even_months]})  # Collect the key business metrics in a tidy table.

# Create formatted string versions for presentation.  
def format_business_value(metric_name, metric_value):  # Define a helper to format percentages, currency, and durations.
    if 'lift' in metric_name.lower():  # Format lift metrics as percentages.
        return f"{metric_value:.2%}"  # Render the number as a percentage.
    if 'revenue' in metric_name.lower() or 'cost' in metric_name.lower():  # Format money metrics as currency.
        return f"${metric_value:,.2f}"  # Render the number as dollars.
    if 'break-even' in metric_name.lower():  # Format the payback period with a readable fallback.
        return 'Not reached' if np.isinf(metric_value) else f"{metric_value:.1f} months"  # Handle non-positive ROI scenarios explicitly.
    return f"{metric_value:,.2f}"  # Format count-like metrics with comma separators.

# Apply the formatting helper to every row.  
business_impact_table['formatted_value'] = business_impact_table.apply(lambda row: format_business_value(row['metric'], row['value']), axis=1)  # Create presentation-ready values for the summary table.

# Print the business-impact table without the raw numeric helper column.  
print(business_impact_table[['metric', 'formatted_value']].to_string(index=False))  # Display the business translation of the experimental lift.

# Print a short interpretation of the payback logic.  
if np.isinf(break_even_months):  # Handle the case where the redesign does not generate positive incremental revenue.
    print(f"\nAt the observed lift of {observed_lift:.4%}, the redesign does not recover its ${implementation_cost:,.0f} build cost because the projected monthly revenue impact is non-positive.")  # Explain the negative-ROI scenario.
else:  # Handle the case where the redesign appears to pay back its implementation cost.
    print(f"\nAt the observed lift of {observed_lift:.4%}, the redesign would recover its ${implementation_cost:,.0f} build cost in about {break_even_months:.1f} months.")  # Explain the positive-ROI scenario.


                              metric formatted_value
              Observed absolute lift          -0.16%
              Observed relative lift          -1.31%
Projected incremental orders / month          -78.91
    Projected revenue impact / month      $-2,761.92
     Projected revenue impact / year     $-33,143.02
                 Implementation cost      $20,000.00
          Break-even period (months)     Not reached

At the observed lift of -0.1578%, the redesign does not recover its $20,000 build cost because the projected monthly revenue impact is non-positive.


## Section 9 — Final Recommendation

### Executive Summary

- The experiment was designed to answer a causal product question: whether a redesigned landing page improves conversion relative to the current experience.
- Before testing outcomes, we validated core experiment integrity by checking for duplicate users and mismatches between assigned group and observed page. This step is essential because causal conclusions are only credible when assignment and exposure align.
- The cleaned data show that the control and treatment groups are roughly balanced, which supports the validity of the randomized comparison.
- Descriptive analysis indicates that the observed conversion-rate difference is small. Time-series inspection should also be reviewed for stability, because early lifts can reflect novelty rather than durable customer value.
- The frequentist two-proportion z-test does not provide strong enough evidence that the treatment outperforms the control. The confidence interval suggests that any true effect is likely small and may plausibly include no meaningful benefit.
- The Bayesian analysis reaches a similar practical conclusion: uncertainty remains substantial, the treatment does not dominate the control with enough confidence, and the expected loss from a wrong launch decision is not negligible.
- Segment-level analysis by day of week is a useful safeguard against hidden heterogeneity. Unless a segment shows a credible, repeatable, and economically meaningful win, the aggregate result should remain the primary decision anchor.
- Business impact modeling reinforces the statistical conclusion: even if the observed lift were taken at face value, the revenue upside must be weighed against implementation cost, operational complexity, and the uncertainty around the estimate.

### Recommendation

**Recommendation: do not launch the new landing page based on the current evidence alone.**

The experiment does not show a convincing, durable, and economically compelling improvement in conversion. The best next step is either to retain the current page or to run a stronger follow-up experiment with a more meaningfully differentiated design.

### What Would Strengthen the Conclusion

- A longer experiment that fully covers multiple weekly cycles and comfortably exceeds the required sample size.
- Additional segmentation dimensions such as device type, geography, traffic source, new versus returning visitors, or customer intent.
- Better instrumentation for intermediate funnel metrics such as click-through, add-to-cart, checkout initiation, and page speed, which can reveal *why* the redesign underperforms or fails to move the final conversion metric.
- A confirmatory follow-up test with a more ambitious design change if the current treatment appears too incremental to generate a detectable business impact.

This recommendation is designed to be understandable by non-technical stakeholders: the current redesign may look directionally interesting, but the evidence is not yet strong enough to justify changing the production experience for all users.


## Section 10 — Appendix & Methodology Notes

### Assumptions Made Throughout the Analysis

- Users were randomly assigned to control and treatment at exposure time.
- After cleaning, each user contributes one independent experimental observation.
- Conversion is measured consistently across both variants without differential logging bias.
- The chosen minimum detectable effect and power assumptions are reasonable for the business context.
- The Beta(1, 1) prior used in the Bayesian section is intentionally weak and is not meant to encode a strong prior business belief.

### Limitations of the Dataset

- The dataset contains only a limited set of user-level variables, which constrains deeper causal diagnostics and segmentation.
- We do not observe richer funnel events, revenue per user, traffic source, device type, or customer history, all of which could materially change interpretation.
- The analysis assumes the experiment was implemented correctly outside the logged fields we can inspect. Hidden issues such as cookie resets, bot traffic, or delayed conversion attribution cannot be ruled out from this file alone.
- Because the notebook centers on a single historical experiment, its conclusions should not be generalized automatically to future redesigns or other user populations.

### What a More Rigorous Production Experiment Would Look Like

At a mature technology company, a production-grade experiment review would typically include pre-registration of the primary metric and decision rule, automated sample ratio mismatch monitoring, bot and fraud filtering, guardrail metrics (for example bounce rate, page latency, refund rate, or customer support contacts), pre-specified segmentation rules, sequential monitoring guidance, and post-experiment rollout analysis. The team would also connect experiment results to long-term value metrics such as retention, gross margin, or lifetime value rather than relying only on short-term conversion.

### References and Further Reading

- Kohavi, Tang, and Xu — *Trustworthy Online Controlled Experiments*.
- Georgiev — *Statistical Methods in Online A/B Testing*.
- Gelman et al. — *Bayesian Data Analysis*.
- Deng, Xu, Kohavi, and Walker — industry papers on metric sensitivity and experimentation platforms.
- Statsmodels documentation for power analysis and proportion tests.
- Plotly documentation for interactive analytical reporting.
